In [4]:
import warnings
warnings.filterwarnings("ignore", category = UserWarning)

In [3]:
import torch
import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader, random_split
from utils import *
from cifar_dla import *
from mnist_cnn import *

### 1. MNIST 베이스 모델 학습

In [3]:
transform = transforms.Compose([
    transforms.ToTensor(), # PyTorch 텐서로 변환
    transforms.Normalize((0.1307,), (0.3081,)) # MNIST 평균 0.1307, 표준편차 0.3081로 정규화
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

val_ratio = 0.2
val_size = int(len(train_dataset) * val_ratio)
train_size = len(train_dataset) - val_size
train_subset, val_subset = random_split(train_dataset, [train_size, val_size], generator = torch.Generator().manual_seed(42))

train_loader = DataLoader(train_subset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_subset, batch_size = 1000, shuffle = False)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

print(f"훈련 데이터 개수: {len(train_subset)}")
print(f"검증 데이터 개수: {len(val_subset)}")
print(f"테스트 데이터 개수: {len(test_dataset)}")

훈련 데이터 개수: 48000
검증 데이터 개수: 12000
테스트 데이터 개수: 10000


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cnn = MNIST_CNN().to(device)
print(cnn)

MNIST_CNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=3136, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)


In [5]:
train_model(cnn, train_loader, val_loader, epochs = 5, mode = "mnist")

Epoch 1/5:   0%|          | 0/750 [00:00<?, ?it/s]

Epoch 1/5: 100%|██████████| 750/750 [00:08<00:00, 90.96it/s, loss=0.1527] 


 Val Accuracy: 98.21

Epoch 2/5: 100%|██████████| 750/750 [00:09<00:00, 80.89it/s, loss=0.0471] 


 Val Accuracy: 98.63

...

Epoch 4/5: 100%|██████████| 750/750 [00:09<00:00, 80.19it/s, loss=0.0231]


 Val Accuracy: 98.44


Epoch 5/5: 100%|██████████| 750/750 [00:09<00:00, 81.75it/s, loss=0.0176] 


 Val Accuracy: 98.85

Training completed. Best Val Acc: 98.94%
     ✅ Best model saved (acc: 98.94%)


MNIST_CNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=3136, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)

In [6]:
evaluate_model(cnn, test_loader, device = device)

98.95

### 2. CIFAR-10 모델 학습 (DLA)

In [11]:
# 데이터셋 로드

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean = (0.4914, 0.4822, 0.4465), # RGB 채널별 평균
        std = (0.2470, 0.2435, 0.2616) # RGB 채널별 표준편차
    )
])

train_dataset = datasets.CIFAR10(root = "./data", train = True, download = True, transform = transform)
test_dataset = datasets.CIFAR10(root = "./data", train = False, download = True, transform = transform)

val_size = int(len(train_dataset) * 0.2)
train_size = len(train_dataset) - val_size

train_subset, val_subset = random_split(
    train_dataset,
    [train_size, val_size],
    generator = torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_subset, batch_size = 64, shuffle = True)
val_loader = DataLoader(val_subset, batch_size = 1000, shuffle = False)
test_loader = DataLoader(test_dataset, batch_size = 1000, shuffle = False)

print(f"훈련 데이터 개수: {len(train_subset)}")
print(f"검증 데이터 개수: {len(val_subset)}")
print(f"테스트 데이터 개수: {len(test_dataset)}")

훈련 데이터 개수: 40000
검증 데이터 개수: 10000
테스트 데이터 개수: 10000


In [12]:
dla = SimpleDLA().to(device)

dla.train()
train_model(dla, train_loader, val_loader, epochs = 20, device = device, mode = "cifar")


Epoch 1/20: 100%|██████████| 625/625 [04:15<00:00,  2.44it/s, loss=1.5314]


 Val Accuracy: 51.72

Epoch 2/20: 100%|██████████| 625/625 [04:14<00:00,  2.45it/s, loss=1.1049]


 Val Accuracy: 61.61

Epoch 3/20: 100%|██████████| 625/625 [04:15<00:00,  2.45it/s, loss=0.8880]


...

Epoch 19/20: 100%|██████████| 625/625 [04:15<00:00,  2.45it/s, loss=0.0586]


 Val Accuracy: 80.77


Epoch 20/20: 100%|██████████| 625/625 [04:16<00:00,  2.44it/s, loss=0.0474]


 Val Accuracy: 80.86

Training completed. Best Val Acc: 81.31%
     ✅ Best model saved (acc: 81.31%)


SimpleDLA(
  (base): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (layer1): Sequential(
    (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (layer2): Sequential(
    (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (layer3): Tree(
    (root): Root(
      (conv): Conv2d(128, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (left_tree): BasicBlock(
      (conv1): Conv2d(32, 64, kernel_size=(3, 3), stride

In [14]:
dla = SimpleDLA()
dla.load_state_dict(torch.load("./training_result/cifar_202603281646.pt"))
evaluate_model(dla, test_loader, device = device)

80.51

### CIFAR-10 모델 학습 (ResNet18, Pre-trained + Fine-tuning)

In [1]:
import resnet18
from utils import *

In [5]:
# 데이터셋 로드

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean = (0.4914, 0.4822, 0.4465), # RGB 채널별 평균
        std = (0.2470, 0.2435, 0.2616) # RGB 채널별 표준편차
    )
])

train_dataset = datasets.CIFAR10(root = "./data", train = True, download = True, transform = transform)
test_dataset = datasets.CIFAR10(root = "./data", train = False, download = True, transform = transform)

val_size = int(len(train_dataset) * 0.2)
train_size = len(train_dataset) - val_size

train_subset, val_subset = random_split(
    train_dataset,
    [train_size, val_size],
    generator = torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_subset, batch_size = 64, shuffle = True)
val_loader = DataLoader(val_subset, batch_size = 1000, shuffle = False)
test_loader = DataLoader(test_dataset, batch_size = 1000, shuffle = False)

print(f"훈련 데이터 개수: {len(train_subset)}")
print(f"검증 데이터 개수: {len(val_subset)}")
print(f"테스트 데이터 개수: {len(test_dataset)}")

훈련 데이터 개수: 40000
검증 데이터 개수: 10000
테스트 데이터 개수: 10000


In [13]:
resnet = resnet18.ResNet18(freeze_backbone= False)

# 학습 대상 파라미터 확인
trainable = sum(p.numel() for p in resnet.parameters() if p.requires_grad)
total = sum(p.numel() for p in resnet.parameters())

print(f"학습 파라미터: {trainable:,}")
print(f"전체 파라미터: {total:,}")

학습 파라미터: 11,173,962
전체 파라미터: 11,173,962


In [14]:
print(resnet)

ResNet18(
  (model): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): Identity()
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv

In [16]:
trained_model = train_model(
    model = resnet,
    train_loader = train_loader,
    val_loader = val_loader,
    epochs = 10,
    lr = 1e-4,
    mode = "cifar_resnet18_finetune",
)

Epoch 1/10: 100%|██████████| 625/625 [02:36<00:00,  3.99it/s, loss=0.7397]


 Val Accuracy: 85.18

Epoch 2/10: 100%|██████████| 625/625 [02:36<00:00,  3.99it/s, loss=0.2909]


 Val Accuracy: 87.45

Epoch 3/10: 100%|██████████| 625/625 [02:35<00:00,  4.01it/s, loss=0.1146]


...

 Val Accuracy: 88.33


Epoch 9/10: 100%|██████████| 625/625 [02:35<00:00,  4.02it/s, loss=0.0341]


 Val Accuracy: 88.81

Epoch 10/10: 100%|██████████| 625/625 [02:35<00:00,  4.01it/s, loss=0.0253]


 Val Accuracy: 89.07
Training completed. Best Val Acc: 89.07%
     ✅ Best model saved (acc: 89.07%)


In [ ]:
# 추가 Fine-tuning 10 epoch 수행
trained_model = train_model(
    model = trained_model,
    train_loader = train_loader,
    val_loader = val_loader,
    epochs = 10,
    lr = 1e-4,
    mode = "cifar_resnet18_finetune",
)

In [18]:
evaluate_model(trained_model, test_loader)

88.68